# Donut — OCR-Free Invoice Field Extraction

This notebook fine-tunes the pretrained **Donut** model (`naver-clova-ix/donut-base`) on invoice images and uses it to extract structured fields directly — no OCR step involved.

**Pipeline:**
```
invoice image → Donut (vision encoder + text decoder) → JSON output
```

**Target fields:**
- `invoice_number`
- `date`
- `total`

---
## 1. Setup

In [ ]:
import os
import json
import re
import random
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    VisionEncoderDecoderConfig,
)
from transformers import get_scheduler
from torch.optim import AdamW
from tqdm.auto import tqdm

# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_NAME   = "naver-clova-ix/donut-base"
DATA_DIR     = Path("../train")      # project invoice images + annotations
IMAGE_DIR    = DATA_DIR               # images live alongside the CSV
LABELS_FILE  = DATA_DIR / "labels.json"
OUTPUT_DIR   = Path("donut_finetuned")

MAX_LENGTH   = 128     # max decoder token length
IMAGE_SIZE   = [1280, 960]  # (height, width) — Donut default
BATCH_SIZE   = 2
NUM_EPOCHS   = 5
LR           = 5e-5
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

# Task prompt that tells the decoder what to produce
TASK_PROMPT  = "<s_invoice>"

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 2. Data Preparation

Your dataset needs a `labels.json` file with entries like:
```json
[
  {
    "file": "invoice_001.jpg",
    "ground_truth": "{\"invoice_number\": \"INV-001\", \"date\": \"2024-01-15\", \"total\": \"1450.00\"}"
  }
]
```

The helper cell below can convert an existing CSV annotations file (from the project dataset) into this format.

In [ ]:
# ── Helper: convert project CSV annotations → labels.json ──────────────────
# Only needed if you do not already have labels.json.
# Adjust CSV_PATH to point at your annotations file.

import csv
from collections import defaultdict

CSV_PATH = DATA_DIR / "_annotations.csv"   # filename,width,height,class,xmin,ymin,xmax,ymax

# Fields we want to keep — class names match the French labels in the project CSV
TARGET_CLASSES = {"Numéro_Facture", "Date_Facturation", "Total_TTC"}
CLASS_TO_KEY   = {"Numéro_Facture":   "invoice_number",
                  "Date_Facturation": "date",
                  "Total_TTC":        "total"}

def build_labels_json(csv_path: Path, out_path: Path) -> None:
    per_image = defaultdict(dict)
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cls = row["class"]
            if cls in TARGET_CLASSES:
                key = CLASS_TO_KEY[cls]
                # Use class name as placeholder — replace with real OCR values if available
                per_image[row["filename"]][key] = f"<{key}_value>"

    records = [
        {"file": fname, "ground_truth": json.dumps(fields)}
        for fname, fields in per_image.items()
    ]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(records, f, indent=2)
    print(f"Wrote {len(records)} records to {out_path}")

if CSV_PATH.exists() and not LABELS_FILE.exists():
    build_labels_json(CSV_PATH, LABELS_FILE)
else:
    print("labels.json already exists or CSV not found — skipping conversion.")

In [ ]:
# ── Dataset class ───────────────────────────────────────────────────────────

class InvoiceDonutDataset(Dataset):
    def __init__(self, records, image_dir, processor, max_length, task_prompt):
        self.records     = records
        self.image_dir   = Path(image_dir)
        self.processor   = processor
        self.max_length  = max_length
        self.task_prompt = task_prompt

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        image  = Image.open(self.image_dir / record["file"]).convert("RGB")

        # Build target string: <task_prompt><s>JSON</s>
        target_sequence = (
            self.task_prompt
            + self.processor.tokenizer.bos_token
            + record["ground_truth"]
            + self.processor.tokenizer.eos_token
        )

        pixel_values = self.processor(
            image, return_tensors="pt"
        ).pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            target_sequence,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)

        # Replace padding token id with -100 so it is ignored in loss
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}


# ── Load records and split ───────────────────────────────────────────────────

with open(LABELS_FILE) as f:
    all_records = json.load(f)

random.seed(42)
random.shuffle(all_records)

split = int(0.8 * len(all_records))
train_records = all_records[:split]
val_records   = all_records[split:]

print(f"Train: {len(train_records)}  |  Val: {len(val_records)}")

---
## 3. Load Pretrained Donut

In [ ]:
processor = DonutProcessor.from_pretrained(MODEL_NAME)
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# ── Add task token to the tokenizer ─────────────────────────────────────────
new_special_tokens = [TASK_PROMPT]
processor.tokenizer.add_special_tokens({"additional_special_tokens": new_special_tokens})
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# ── Align processor image size with model config ─────────────────────────────
processor.feature_extractor.size = {"height": IMAGE_SIZE[0], "width": IMAGE_SIZE[1]}
processor.feature_extractor.do_align_long_axis = False

model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(TASK_PROMPT)
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.eos_token_id
model.config.max_length             = MAX_LENGTH

model.to(DEVICE)
print("Model loaded.")

---
## 4. Fine-Tuning

In [ ]:
train_dataset = InvoiceDonutDataset(train_records, IMAGE_DIR, processor, MAX_LENGTH, TASK_PROMPT)
val_dataset   = InvoiceDonutDataset(val_records,   IMAGE_DIR, processor, MAX_LENGTH, TASK_PROMPT)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=NUM_EPOCHS * len(train_loader),
)

best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]"):
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels       = batch["labels"].to(DEVICE)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    avg_train = train_loss / len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [val]"):
            pixel_values = batch["pixel_values"].to(DEVICE)
            labels       = batch["labels"].to(DEVICE)
            outputs      = model(pixel_values=pixel_values, labels=labels)
            val_loss    += outputs.loss.item()

    avg_val = val_loss / len(val_loader)
    print(f"Epoch {epoch} — train loss: {avg_train:.4f}  |  val loss: {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        print(f"  ✓ Saved best model (val loss: {best_val_loss:.4f})")

print("Fine-tuning complete.")

---
## 5. Inference on a Single Invoice Image

In [ ]:
def extract_invoice_fields(image_path: str, checkpoint_dir: str = str(OUTPUT_DIR)) -> dict:
    """Run fine-tuned Donut on a single invoice image and return extracted fields."""
    proc  = DonutProcessor.from_pretrained(checkpoint_dir)
    mdl   = VisionEncoderDecoderModel.from_pretrained(checkpoint_dir).to(DEVICE)
    mdl.eval()

    image = Image.open(image_path).convert("RGB")
    pixel_values = proc(image, return_tensors="pt").pixel_values.to(DEVICE)

    task_token_id = proc.tokenizer.convert_tokens_to_ids(TASK_PROMPT)
    decoder_input = torch.full((1, 1), task_token_id, dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        generated = mdl.generate(
            pixel_values,
            decoder_input_ids=decoder_input,
            max_length=MAX_LENGTH,
            early_stopping=True,
            pad_token_id=proc.tokenizer.pad_token_id,
            eos_token_id=proc.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=1,
            bad_words_ids=[[proc.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    sequence = proc.batch_decode(generated.sequences)[0]
    # Strip special tokens and task prompt
    sequence = sequence.replace(proc.tokenizer.eos_token, "").replace(proc.tokenizer.pad_token, "")
    sequence = re.sub(r"<.*?>", "", sequence).strip()

    try:
        result = json.loads(sequence)
    except json.JSONDecodeError:
        result = {"raw_output": sequence}

    return result


# ── Example ──────────────────────────────────────────────────────────────────
TEST_IMAGE = str(IMAGE_DIR / all_records[0]["file"])   # replace with any invoice path
result = extract_invoice_fields(TEST_IMAGE)
print(json.dumps(result, indent=2))

---
## 6. Evaluation

We compute **exact match** and **partial match** (case-insensitive, stripped) for each target field across the validation set.

In [ ]:
def normalise(value: str) -> str:
    """Lowercase, strip whitespace and common currency symbols."""
    return re.sub(r"[\$€£,]", "", str(value)).strip().lower()


def evaluate_on_val(val_records, image_dir, checkpoint_dir):
    proc = DonutProcessor.from_pretrained(checkpoint_dir)
    mdl  = VisionEncoderDecoderModel.from_pretrained(checkpoint_dir).to(DEVICE)
    mdl.eval()

    exact_total  = {"invoice_number": 0, "date": 0, "total": 0}
    partial_total = {"invoice_number": 0, "date": 0, "total": 0}
    n = len(val_records)

    for record in tqdm(val_records, desc="Evaluating"):
        gt = json.loads(record["ground_truth"])
        pred = extract_invoice_fields(str(Path(image_dir) / record["file"]), checkpoint_dir)

        for field in exact_total:
            gt_val   = normalise(gt.get(field, ""))
            pred_val = normalise(pred.get(field, ""))
            if gt_val == pred_val:
                exact_total[field] += 1
            if gt_val and gt_val in pred_val:
                partial_total[field] += 1

    print(f"\nResults over {n} validation samples:\n")
    print(f"{'Field':<20} {'Exact Match':>12} {'Partial Match':>14}")
    print("-" * 48)
    for field in exact_total:
        em  = exact_total[field]  / n * 100
        pm  = partial_total[field] / n * 100
        print(f"{field:<20} {em:>11.1f}%  {pm:>13.1f}%")


evaluate_on_val(val_records, IMAGE_DIR, str(OUTPUT_DIR))

---
## 7. YOLO Benchmark

We compare Donut's field-level accuracy against the best YOLO detection checkpoint.
YOLO works at the **bounding-box** level — it predicts *where* a field is, not *what* it says.
We therefore measure **field detection rate**: the fraction of validation images on which YOLO
returns at least one box for each target field class (`Numéro_Facture`, `Date_Facturation`,
`Total_TTC`), using a confidence threshold of 0.25.

This gives a fair apples-to-apples comparison: Donut exact/partial match vs YOLO detection rate.

In [ ]:
# ── YOLO Benchmark ───────────────────────────────────────────────────────────
# Requires: pip install ultralytics

from pathlib import Path

# Best YOLO checkpoint trained on the same invoice dataset
YOLO_CHECKPOINT = (
    Path.cwd().parent / "YOLO_method" / "runs" / "invoice_fields_960_best2" / "weights" / "best.pt"
)
YOLO_CONF       = 0.25   # confidence threshold

# Mapping from YOLO class name → Donut field key
YOLO_CLASS_TO_FIELD = {
    "Numéro_Facture":   "invoice_number",
    "Date_Facturation": "date",
    "Total_TTC":        "total",
}


def yolo_field_detection_rate(val_records, image_dir, checkpoint, conf=0.25):
    """Return per-field detection rates for YOLO on the validation set."""
    try:
        from ultralytics import YOLO
    except ImportError:
        raise ImportError("Install ultralytics: pip install ultralytics")

    if not Path(checkpoint).exists():
        raise FileNotFoundError(f"YOLO checkpoint not found: {checkpoint}")

    yolo = YOLO(str(checkpoint))

    detected = {f: 0 for f in YOLO_CLASS_TO_FIELD.values()}
    n = len(val_records)

    for record in tqdm(val_records, desc="YOLO benchmark"):
        img_path = str(Path(image_dir) / record["file"])
        results  = yolo.predict(source=img_path, conf=conf, verbose=False)

        found_classes = set()
        for r in results:
            for cls_id in r.boxes.cls.int().tolist():
                cls_name = r.names[cls_id]
                if cls_name in YOLO_CLASS_TO_FIELD:
                    found_classes.add(YOLO_CLASS_TO_FIELD[cls_name])

        for field in detected:
            if field in found_classes:
                detected[field] += 1

    return {field: detected[field] / n * 100 for field in detected}, n


def benchmark_comparison(val_records, image_dir, donut_checkpoint, yolo_checkpoint):
    """Print a side-by-side table of Donut vs YOLO accuracy metrics."""
    # ── Donut metrics ────────────────────────────────────────────────────────
    proc = DonutProcessor.from_pretrained(donut_checkpoint)
    mdl  = VisionEncoderDecoderModel.from_pretrained(donut_checkpoint).to(DEVICE)
    mdl.eval()

    exact_hits   = {"invoice_number": 0, "date": 0, "total": 0}
    partial_hits = {"invoice_number": 0, "date": 0, "total": 0}
    n = len(val_records)

    for record in tqdm(val_records, desc="Donut evaluation"):
        gt   = json.loads(record["ground_truth"])
        pred = extract_invoice_fields(str(Path(image_dir) / record["file"]), donut_checkpoint)
        for field in exact_hits:
            gt_v   = normalise(gt.get(field, ""))
            pred_v = normalise(pred.get(field, ""))
            if gt_v == pred_v:
                exact_hits[field] += 1
            if gt_v and gt_v in pred_v:
                partial_hits[field] += 1

    donut_exact   = {f: exact_hits[f]   / n * 100 for f in exact_hits}
    donut_partial = {f: partial_hits[f] / n * 100 for f in partial_hits}

    # ── YOLO metrics ─────────────────────────────────────────────────────────
    yolo_detect, _ = yolo_field_detection_rate(val_records, image_dir, yolo_checkpoint)

    # ── Print table ──────────────────────────────────────────────────────────
    print(f"\nBenchmark — {n} validation images\n")
    header = f"{'Field':<20} {'Donut Exact':>12} {'Donut Partial':>14} {'YOLO Detected':>14}"
    print(header)
    print("-" * len(header))
    for field in ("invoice_number", "date", "total"):
        print(
            f"{field:<20} {donut_exact[field]:>11.1f}%"
            f"  {donut_partial[field]:>12.1f}%"
            f"  {yolo_detect[field]:>12.1f}%"
        )


benchmark_comparison(
    val_records,
    IMAGE_DIR,
    donut_checkpoint=str(OUTPUT_DIR),
    yolo_checkpoint=YOLO_CHECKPOINT,
)
